In [4]:
%%time
from tqdm import tqdm
from datetime import timedelta
# from datetime import datetime
import joblib
import cftime
import numpy as np
import pandas as pd
import xarray as xr
import lightgbm as lgb
import gc
import os, sys
sys.path.insert(0,'/home/mingyu_seong/Works/func/python')
import mg

CPU times: user 21 μs, sys: 0 ns, total: 21 μs
Wall time: 23.6 μs


### Read Annual data

In [5]:
#--- Input period
sdate = '2001-01-01'
edate = '2100-12-30'

#--- Clim period
clim_syr = 2001
clim_eyr = 2020

#--- Variables for annual data
ann_feature_names_ordered = ['T_clim', 'P_clim', 'SM_clim', 'SW_clim',
                             'T_anom', 'P_anom', 'SM_anom', 'SW_anom']

#--- Directory
idir_ann = '../../../_nc/input/HadGEM3-RA/background/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/'
idir_ann_clim = idir_ann + 'clim/'
idir_ann_anom = idir_ann + 'anom/'

#--- Input files: Climatology
fin_ann_clim_tas  = 'tas_mon_ERA5_V0_KOR_01deg_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
fin_ann_clim_prcp = 'pr_mon_GPM_L3_KOR_01deg_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
fin_ann_clim_sm   = 'mrso_mon_ERA5_V0_KOR_01deg_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
fin_ann_clim_rsds = 'rsds_mon_MODIS_C62_KOR_01deg_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'

### Making 8-day Reco and then removing background changes

In [ ]:
%%time
# CPU times: user 12h 12min 50s, sys: 2min 38s, total: 12h 15min 29s
# Wall time: 1h 41min 58s

#--- Random seeds (42, 500, 123, 1000, 5473)
rs = 42

#--- Variables for Reco variation
feature_names_ordered = [
    'T_clim', 'P_clim', 'SM_clim', 'SW_clim', 'Elev',
    'T_anom', 'P_anom', 'SM_anom', 'SW_anom', 
    'cos_lon', 'sin_lon', 'cos_lat', 'sin_lat', 'cal_day'
]

#--- 8-day dates
date_8days = []
for y in range(int(sdate.split('-')[0]), int(edate.split('-')[0])+1):
    for m in range(1, 13):
        for d in [1, 9, 17, 25]:
            date_8days.append(cftime.Datetime360Day(y, m, d, 0, 0, 0, 0, has_year_zero=True))

#--- Directories
idir = '/home/share/KCP/2025_final_input/_nc/input/r' + str(clim_syr)[2:4] + str(clim_eyr)[2:4] + '/HadGEM3-RA/'
idir_clim = idir + 'clim/'
idir_raw  = idir + 'auxiliary/'

for isce in tqdm(['rcp26', 'rcp45', 'rcp60', 'rcp85']):
    # 1) Annual Reco (Background)
    fin_ann_anom_tas  = 'tas_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
    fin_ann_anom_prcp = 'pr_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
    fin_ann_anom_sm   = 'mrso_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
    fin_ann_anom_rsds = 'rsds_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_mon_' + str(sdate[0:4]) + '01-' + str(edate[0:4]) + '12.nc'
    ann_data_files = {
        'T_clim':  (idir_ann_clim, fin_ann_clim_tas,  'clim'),
        'P_clim':  (idir_ann_clim, fin_ann_clim_prcp, 'clim'),
        'SM_clim': (idir_ann_clim, fin_ann_clim_sm,   'clim'),
        'SW_clim': (idir_ann_clim, fin_ann_clim_rsds, 'clim'),
        'T_anom':  (idir_ann_anom, fin_ann_anom_tas,  'anom'),
        'P_anom':  (idir_ann_anom, fin_ann_anom_prcp, 'anom'),
        'SM_anom': (idir_ann_anom, fin_ann_anom_sm,   'anom'),
        'SW_anom': (idir_ann_anom, fin_ann_anom_rsds, 'anom'),
    }
    raw_data_arrays = {}
    for var_name, (path, filename, da_name) in ann_data_files.items():
        full_path = os.path.join(path, filename)
        try:
            with xr.open_dataset(full_path) as ncid:
                da = ncid[da_name].copy()
                raw_data_arrays[var_name] = da.groupby('time.year').mean()  # Annual mean
        except FileNotFoundError:
            print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
            sys.exit(1) # 파일이 없으면 스크립트 중단
    ann_features_ds = xr.Dataset()
    for feature_name in ann_feature_names_ordered:
        da = raw_data_arrays[feature_name]
        ann_features_ds[feature_name] = da
 
    # 2) Reco variation (removing background)
    idir_anom = idir + 'anom/' + isce + '/'
    odir = '../../../_nc/prediction/HadGEM3-RA/Reco/variation/' + isce + '/'
    os.makedirs(odir, exist_ok=True) 
    
    #--- Input files (Climatology & Anomaly)
    fin_anom_tas  = 'tas_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_anom_pr   = 'pr_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_anom_sm   = 'mrso_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_anom_rsds = 'rsds_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_anom_co2  = 'co2_KOR-11_HadGEM2-AO_' + isce + '_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_clim_tas  = 'tas_day_ERA5_V0_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_clim_pr   = 'pr_day_GPM_L3_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_clim_sm   = 'mrso_day_ERA5_V0_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_clim_rsds = 'rsds_day_MODIS_C62_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_clim_co2  = 'co2_day_noaa_gml_KOR_01deg_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_elev = 'elev_KOR-11_srtm30plus_v11_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_loc  = 'lonlat_KOR-11_HadGEM2-AO_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    fin_cday = 'calday_KOR-11_HadGEM2-AO_r2i1p1_HadGEM3-RA_v1_day_' + sdate.replace('-','') + '-' + edate.replace('-','') + '.nc'
    
    data_files = {
        'T_clim':  (idir_clim, fin_clim_tas, 'clim'),
        'P_clim':  (idir_clim, fin_clim_pr,  'clim'),
        'SM_clim': (idir_clim, fin_clim_sm,  'clim'),
        'SW_clim': (idir_clim, fin_clim_rsds,'clim'),
        'T_anom':  (idir_anom, fin_anom_tas, 'anom'),
        'P_anom':  (idir_anom, fin_anom_pr,  'anom'),
        'SM_anom': (idir_anom, fin_anom_sm,  'anom'),
        'SW_anom': (idir_anom, fin_anom_rsds,'anom'),
        'Elev':    (idir_raw, fin_elev,'elev'), 
        'cal_day': (idir_raw, fin_cday,'calday'),
        'cos_lon': (idir_raw, fin_loc, 'coslon'),
        'sin_lon': (idir_raw, fin_loc, 'sinlon'),
        'cos_lat': (idir_raw, fin_loc, 'coslat'),
        'sin_lat': (idir_raw, fin_loc, 'sinlat'), }
    
    #--- Read data
    raw_data_arrays = {}
    for var_name, (path, filename, da_name) in data_files.items():
        full_path = os.path.join(path, filename)
        try:
            with xr.open_dataset(full_path) as ncid:
                da = ncid[da_name]
                if 'time' in da.dims:
                    da = da.sel(time=slice(sdate, edate))
                    new_times = []
                    for t in da.coords['time'].values:
                        new_time = cftime.Datetime360Day(t.year, t.month, t.day, 0, 0, 0)
                        new_times.append(new_time)
                    da = da.assign_coords(time=new_times)
                    
                #--- 8-day average
                da_accum = []
                for t_8day in date_8days:
                    start_date = cftime.Datetime360Day(t_8day.year, t_8day.month, t_8day.day)
                    if start_date.day == 25:
                        end_date = start_date + timedelta(days=5)             # 6-day mean
                    else:
                        end_date = start_date + timedelta(days=7)             # 8-day mean
                    data_8day = da.sel(time=slice(start_date, end_date))
                    if (var_name == "Elev") or (var_name == "cal_day") or    \
                       (var_name == "sin_lon") or (var_name == "sin_lat") or \
                       (var_name == "cos_lon") or (var_name == "cos_lat"):
                        data_accum = data_8day.sel(time=start_date)
                    else:
                        data_accum = data_8day.mean(dim='time',skipna=False)
                    da_accum.append(data_accum)
    
                #--- Add 8day time info
                da_accum = xr.concat(da_accum, dim='time')
                da_accum['time'] = date_8days
                raw_data_arrays[var_name] = da_accum
                
        except FileNotFoundError:
            print(f"  오류: 파일이 존재하지 않습니다: {full_path}")
            sys.exit(1) # 파일이 없으면 스크립트 중단
    
    #--- input & label
    features_ds = xr.Dataset()
    for feature_name in feature_names_ordered:
        da = raw_data_arrays[feature_name].copy()
        if feature_name in ann_feature_names_ordered:
            da.data = da.data - np.repeat(ann_features_ds[feature_name].data, 48)[:, np.newaxis, np.newaxis]  # variation
        features_ds[feature_name] = da

    #--- Combined inputs
    # print(f"데이터셋으로 통합 및 스태킹 중...")
    combined_features_da = features_ds.to_array(dim='variable')

    #--- Inference by 12 models
    for ifold in np.arange(12):
        #--- Initialization
        train_start  = None ; train_end  = None
        train_start0 = None ; train_end0 = None
        train_start1 = None ; train_end1 = None
        valid_start  = None ; valid_end  = None
        valid_start0 = None ; valid_end0 = None
        valid_start1 = None ; valid_end1 = None
        test_start   = None ; test_end   = None
        
        # Test: 17-yr / Validation: 5-yr / Test: 2-yr  (Same as FFNN)
        if ifold == 0:
            train_start = '2003' ; train_end = '2019'
            valid_start = '2020' ; valid_end = '2024'
            test_start  = '2001' ; test_end  = '2002'
        elif ifold == 1:
            train_start  = '2005' ; train_end  = '2021'
            valid_start0 = '2022' ; valid_end0 = '2024'
            valid_start1 = '2001' ; valid_end1 = '2002'
            test_start   = '2003' ; test_end   = '2004'
        elif ifold == 2:   
            train_start  = '2005' ; train_end  = '2023'
            valid_start0 = '2024' ; valid_end0 = '2024'
            valid_start1 = '2001' ; valid_end1 = '2004'
            test_start   = '2005' ; test_end   = '2006'
        elif ifold == 3:   
            train_start0 = '2001' ; train_end0 = '2001'
            train_start1 = '2009' ; train_end1 = '2024'
            valid_start  = '2002' ; valid_end  = '2006'
            test_start   = '2007' ; test_end   = '2008'
        elif ifold == 4:   
            train_start0 = '2001' ; train_end0 = '2003'
            train_start1 = '2011' ; train_end1 = '2024'
            valid_start  = '2004' ; valid_end  = '2008'
            test_start   = '2009' ; test_end   = '2010'
        elif ifold == 5:   
            train_start0 = '2001' ; train_end0 = '2005'
            train_start1 = '2013' ; train_end1 = '2024'
            valid_start  = '2006' ; valid_end  = '2010'
            test_start   = '2011' ; test_end   = '2012'
        elif ifold == 6:   
            train_start0 = '2001' ; train_end0 = '2007'
            train_start1 = '2015' ; train_end1 = '2024'
            valid_start  = '2008' ; valid_end  = '2012'
            test_start   = '2013' ; test_end   = '2014'
        elif ifold == 7:   
            train_start0 = '2001' ; train_end0 = '2009'
            train_start1 = '2017' ; train_end1 = '2024'
            valid_start  = '2010' ; valid_end  = '2014'
            test_start   = '2015' ; test_end   = '2016'
        elif ifold == 8:   
            train_start0 = '2001' ; train_end0 = '2011'
            train_start1 = '2019' ; train_end1 = '2024'
            valid_start  = '2012' ; valid_end  = '2016'
            test_start   = '2017' ; test_end   = '2018'
        elif ifold == 9:   
            train_start0 = '2001' ; train_end0 = '2013'
            train_start1 = '2021' ; train_end1 = '2024'
            valid_start  = '2014' ; valid_end  = '2018'
            test_start   = '2019' ; test_end   = '2020'
        elif ifold == 10:   
            train_start0 = '2001' ; train_end0 = '2015'
            train_start1 = '2023' ; train_end1 = '2024'
            valid_start  = '2016' ; valid_end  = '2020'
            test_start   = '2021' ; test_end   = '2022'
        elif ifold == 11:   
            train_start = '2001' ; train_end = '2017'
            valid_start = '2018' ; valid_end = '2022'
            test_start  = '2023' ; test_end  = '2024'

        test_features_da = combined_features_da
        
        stacked_test_features_list = []
        for i, feature_name in enumerate(feature_names_ordered):
            stacked_test_feature_array = test_features_da[i,:,:,:].stack(sample=('time', 'lat', 'lon')).values
            stacked_test_features_list.append(stacked_test_feature_array)
        
        #--- Separate data (train / validation / test)
        test_x = np.transpose(np.vstack(stacked_test_features_list))
        
        #--- Load model structure
        model_path = './lightGBM/lgbm_model_' + test_start.replace('-','') + \
                     '-' + test_end.replace('-','') + \
                     '_fold' + str(ifold).zfill(2) + \
                     '_seed' + str(rs) + '.joblib'
        model = joblib.load(model_path)
    
        #--- Inference
        y_pred = model.predict(test_x)
        pred = y_pred.reshape(len(combined_features_da[0,:,:,:].time.data),
                              len(combined_features_da[0,:,:,:].lat.data),
                              len(combined_features_da[0,:,:,:].lon.data))
        pred[np.isclose(pred, pred[0,0,0] , atol=1e-6 )] = np.nan

        #--- Save
        pred_data = xr.DataArray(pred,
                                dims=['time', 'lat', 'lon'],
                                coords={'time': date_8days,
                                        'lat': combined_features_da[0,:,:,:].lat.data, 
                                        'lon': combined_features_da[0,:,:,:].lon.data},
                                attrs={'long_name': 'Predicted Reco variation',
                                       'units': 'gC/m^2/8days'}) 
        fou = 'predicted_Reco_8day_LGBM_' + isce + '_' + sdate.replace('-','') + \
              '-' + edate.replace('-','') + '_fold' + str(ifold).zfill(2) + '_seed' + str(rs) + '.nc'
        ds = xr.Dataset()
        ds['reco'] = pred_data
        ds['lon'].attrs['long_name'] = "longitude"
        ds['lat'].attrs['long_name'] = "latitude"
        ds['lon'].attrs['axis'] = "X"
        ds['lat'].attrs['axis'] = "Y"
        ds.to_netcdf(odir + fou, encoding={'time': {'dtype':'int32','calendar':'360_day'}})
        del test_x, pred_data, ds, model
    
    #--- Ensemble mean
    fin = 'predicted_Reco_8day_LGBM_' + isce + '_' + sdate.replace('-','') + \
          '-' + edate.replace('-','') + '_fold*' + '_seed' + str(rs) + '.nc'
    fou = 'predicted_Reco_8day_LGBM_' + isce + '_' + sdate.replace('-','') + \
              '-' + edate.replace('-','') + '_ensmean_seed' + str(rs) + '.nc'
    mg.cdo('ensmean', input=[os.path.join(odir, f) for f in mg.get_files(odir, fin)], output=odir+fou,options='-s -L -O' )